# Smoke Test — the go/no-go gate

**Asch in Silicon** · [repo](https://github.com/LihanCanCode/Collective-Cognitive-Error)

Answers the one question that can kill the project before anything is built on top of it:
**do these models conform to a wrong majority at a measurable rate at all?**

### Before you run

Kaggle sidebar → **Settings**:
- **Accelerator:** `GPU T4 x2`
- **Internet:** `On`  (required — needs to clone the repo and download the model)

Expect **10–20 min**, most of it model download.

### Reading the result

| Signal | Meaning |
|---|---|
| baseline error at n=0 **< 10%** | items are easy enough that a wrong answer under pressure means conformity, not ignorance |
| conformity rate **5–70%** | **PASS** — measurable band, proceed to the full grid |
| conformity **~0%** | items too easy or model too independent → raise difficulty |
| conformity **~100%** | no independent judgement left → lower difficulty or change model |

**Do not skip Cell 5.** The conformity number is meaningless if confederates broke character or the
naive agent answered in a format the parser missed — and that failure looks exactly like a clean result.

## Cell 1 — check the environment

Nothing to install. The gate runs on **`transformers`**, which Kaggle already ships.

That is deliberate: the gate is only ~250 generations, so vLLM's throughput buys nothing, while a
pinned vLLM hard-fails on any model config newer than itself — `vllm==0.6.3` cannot parse Qwen2.5's
`rope_scaling` and dies on a bare `AssertionError`. vLLM returns for the full grid, where
throughput is the entire point.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch, transformers

print("transformers", transformers.__version__, "| torch", torch.__version__)
print("CUDA devices:", torch.cuda.device_count())
assert torch.cuda.is_available(), "No GPU. Set Accelerator to 'GPU T4 x2' in the sidebar."

## Cell 2 — get the code and verify the clone

Every path below is absolute and anchored to `REPO_DIR`, so nothing depends on the notebook's
working directory. The assertion is there because a partial or failed clone otherwise shows up
several cells later as a confusing import error.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/LihanCanCode/Collective-Cognitive-Error.git"
REPO_DIR = Path("/kaggle/working/repo")

!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}

expected = ["scripts/run_smoke.py", "scripts/make_smoke_bank.py", "src/asch/runner.py"]
missing = [p for p in expected if not (REPO_DIR / p).exists()]
assert not missing, f"clone incomplete, missing: {missing}"

print("clone OK")
print(subprocess.run(["git", "log", "--oneline", "-1"], cwd=REPO_DIR,
                     capture_output=True, text=True).stdout.strip())

## Cell 3 — verify the pipeline offline

A full end-to-end run on the mock backend: no GPU, no network, no model. If this does not print
`PASS`, the harness itself is broken and no GPU time should be spent on it.

In [ ]:
!python {REPO_DIR}/scripts/make_smoke_bank.py
!python {REPO_DIR}/scripts/run_smoke.py --backend mock --out {REPO_DIR}/results/mock_check.jsonl

## Cell 4 — run the gate

50 items × {n=0 control, n=3 unanimous-wrong} = 100 trials, ~250 generations. n=3 is where human
conformity peaks in Asch (32%).

`device_map="auto"` shards the 7B across both T4s — a 7B in fp16 is ~15 GB and will not fit on one.
Expect **15–30 min**, most of it the model download.

Resumable: if the session dies, just re-run this cell — completed trials are skipped.

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"
RESULTS = REPO_DIR / "results" / "smoke_qwen7b.jsonl"

!python {REPO_DIR}/scripts/run_smoke.py --backend hf --model {MODEL} --out {RESULTS}

## Cell 5 — read five transcripts by hand

Not optional. You are checking three things:
1. Did each confederate actually assert its assigned answer? (`complied=True`)
2. Did the naive agent answer in the `Answer: X` format? (`valid=True`)
3. Does the naive agent's reasoning look like real deliberation, or is it degenerate?

In [ ]:
import json

assert RESULTS.exists(), (
    f"{RESULTS} not found — Cell 4 did not produce results. Scroll up and read its error "
    "output before continuing."
)

records = [json.loads(line) for line in RESULTS.open() if line.strip()]
critical = [r for r in records if r["n_confederates"] == 3]
print(f"{len(records)} trials total, {len(critical)} critical\n")

for rec in critical[:5]:
    print("=" * 95)
    print(
        f"stance={rec['stance']}  answer={rec['answer']}  correct={rec['correct_answer']}  "
        f"majority={rec['majority_answer']}  valid={rec['valid']}"
    )
    for turn in rec.get("transcript", []):
        who = turn["role"]
        extra = (
            f" (assigned {turn['assigned_answer']}, complied={turn['complied']})"
            if who == "confederate"
            else ""
        )
        print(f"\n--- {who}{extra} ---")
        print(turn["text"][:400])

## Cell 6 — diagnostics

Full breakdown: **per-subtype baseline accuracy** (which items are too hard) and the **discard
split** (confederate character-breaks vs parse failures — different problems, different fixes).

Runs on the saved JSONL in seconds, no GPU. **Report this output back.**

In [ ]:
!python {REPO_DIR}/scripts/diagnose.py {RESULTS}

## Cell 7 — save results off the session

Kaggle wipes everything outside `/kaggle/working` when the session ends. Either **Save Version**
(persists `/kaggle/working` as notebook output) or download the JSONL from the file browser.

In [ ]:
import shutil

dest = Path("/kaggle/working") / RESULTS.name
shutil.copy(RESULTS, dest)
print(f"saved -> {dest}  ({dest.stat().st_size / 1024:.0f} KB)")

## Optional A — the faithful-Asch arm (`--confederate-style bare`)

Asch's confederates stated a line and gave **no** reasoning. Ours write persuasive (fabricated)
arguments, which is stronger pressure than Asch's and mixes conformity with being argued into a
position.

`bare` confederates say only `Answer: X`. It needs **no model call for the confederates**, so this
run is ~4× faster than the justified one, and compliance is guaranteed by construction.

Comparing the two conformity rates separates *"I agreed because everyone agreed"* from *"I agreed
because the argument sounded plausible"*. Run it — it is nearly free.

In [ ]:
BARE_RESULTS = REPO_DIR / "results" / "smoke_qwen7b_bare.jsonl"

!python {REPO_DIR}/scripts/run_smoke.py --backend hf --model {MODEL} \
    --confederate-style bare --out {BARE_RESULTS}

!python {REPO_DIR}/scripts/diagnose.py {BARE_RESULTS}

## Optional B — a second model

If Qwen-7B fails the gate, the fastest diagnostic is a second model: a model-specific quirk looks
very different from a genuine ceiling in the item bank. All ungated, no HF token needed.

Re-run Cells 5–7 afterwards to inspect the new results.

In [ ]:
# mistralai/Mistral-7B-Instruct-v0.3
# google/gemma-2-9b-it
# Qwen/Qwen2.5-1.5B-Instruct    <- smaller and fast: expect MORE conformity if the effect is real

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RESULTS = REPO_DIR / "results" / "smoke_qwen1_5b.jsonl"

!python {REPO_DIR}/scripts/run_smoke.py --backend hf --model {MODEL} --out {RESULTS}